# 📦 Day 01a: NoSQL Databases — Document, Key-Value, Column, Graph

---

## 🎯 What You'll Master Today
- 4 types of NoSQL databases
- SQL vs NoSQL tradeoffs
- Data modeling patterns for each type
- When to use which database

---

```
╔══════════════════════════════════════════════════════════════════════╗
║                   NoSQL DATABASE TYPES                               ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. KEY-VALUE     │ Redis, DynamoDB, Memcached                      ║
║     key → value   │ Fastest. Caching, sessions, counters.           ║
║                                                                      ║
║  2. DOCUMENT      │ MongoDB, CouchDB, Firestore                    ║
║     key → JSON    │ Flexible schema. Content, catalogs, profiles.   ║
║                                                                      ║
║  3. COLUMN-FAMILY │ Cassandra, HBase, ScyllaDB                     ║
║     row → columns │ Write-heavy. Time-series, IoT, logs.           ║
║                                                                      ║
║  4. GRAPH         │ Neo4j, Amazon Neptune, ArangoDB                 ║
║     nodes + edges │ Relationships. Social networks, fraud, recs.   ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. Key-Value Store Simulation (Redis-like)
# ============================================

import time, json

class KeyValueStore:
    def __init__(self):
        self.store = {}
        self.ttl = {}  # time-to-live
    
    def set(self, key, value, ttl_seconds=None):
        self.store[key] = value
        if ttl_seconds:
            self.ttl[key] = time.time() + ttl_seconds
    
    def get(self, key):
        if key in self.ttl and time.time() > self.ttl[key]:
            del self.store[key]
            del self.ttl[key]
            return None
        return self.store.get(key)
    
    def incr(self, key):
        self.store[key] = self.store.get(key, 0) + 1
        return self.store[key]

kv = KeyValueStore()

# Session storage
kv.set("session:user123", {"name": "Alice", "role": "admin"}, ttl_seconds=3600)
# Counter
kv.incr("page_views:/home")
kv.incr("page_views:/home")
kv.incr("page_views:/home")
# Cache
kv.set("cache:product:42", {"name": "Widget", "price": 29.99}, ttl_seconds=300)

print("  Key-Value Store (Redis-like):")
print(f"    session:user123     → {kv.get('session:user123')}")
print(f"    page_views:/home    → {kv.get('page_views:/home')}")
print(f"    cache:product:42    → {kv.get('cache:product:42')}")
print("\n  💡 O(1) get/set. No queries, no joins. Pure speed.")

In [ ]:
# ============================================
# 2. Document Store Simulation (MongoDB-like)
# ============================================

class DocumentDB:
    def __init__(self):
        self.collections = {}
    
    def insert(self, collection, doc):
        self.collections.setdefault(collection, []).append(doc)
    
    def find(self, collection, query):
        results = []
        for doc in self.collections.get(collection, []):
            if all(doc.get(k) == v for k, v in query.items()):
                results.append(doc)
        return results

db = DocumentDB()

# Documents with different schemas (flexible!)
db.insert("users", {
    "_id": 1, "name": "Alice", "email": "alice@example.com",
    "address": {"city": "NYC", "zip": "10001"},  # nested!
    "orders": ["ord_1", "ord_2"],  # embedded array!
})
db.insert("users", {
    "_id": 2, "name": "Bob", "email": "bob@example.com",
    "phone": "+1234567890",  # different fields OK!
})

print("  Document Store (MongoDB-like):")
for doc in db.collections["users"]:
    print(f"    {json.dumps(doc, indent=6)}\n")

print("  Query: find({name: 'Alice'})")
results = db.find("users", {"name": "Alice"})
print(f"    → {results[0]['name']}, {results[0]['address']['city']}")

print("\n  💡 Schema-flexible. Embed related data. Avoid JOINs.")
print("  Denormalization: embed orders in user vs separate collection")

In [ ]:
# ============================================
# 3. Column-Family Store (Cassandra-like)
# ============================================

print("  Column-Family Store (Cassandra-like):\n")

# Cassandra data model: partition key + clustering columns
# Optimized for: write-heavy, time-series, wide rows

sensor_data = {
    # partition_key (sensor_id) → sorted by clustering key (timestamp)
    "sensor_001": [
        {"ts": "2024-01-01T00:00", "temp": 22.5, "humidity": 45},
        {"ts": "2024-01-01T00:01", "temp": 22.6, "humidity": 44},
        {"ts": "2024-01-01T00:02", "temp": 22.4, "humidity": 46},
    ],
    "sensor_002": [
        {"ts": "2024-01-01T00:00", "temp": 18.1, "humidity": 60},
        {"ts": "2024-01-01T00:01", "temp": 18.3, "humidity": 59},
    ],
}

print("  Table: sensor_readings")
print("  PRIMARY KEY (sensor_id, timestamp)")
print("  ─ partition_key: sensor_id (distributes data across nodes)")
print("  ─ clustering_key: timestamp (sorts within partition)\n")

for sensor, readings in sensor_data.items():
    print(f"  Partition [{sensor}]:")
    for r in readings:
        print(f"    {r['ts']} | temp={r['temp']} | humidity={r['humidity']}")

print("\n  Efficient query: WHERE sensor_id='sensor_001' AND ts > '2024-01-01T00:01'")
print("  Inefficient: WHERE temp > 22 (requires full scan across all partitions!)")
print("\n  💡 Model tables around query patterns. One table per query.")

In [ ]:
# ============================================
# 4. Graph Database (Neo4j-like)
# ============================================

from collections import defaultdict

class GraphDB:
    def __init__(self):
        self.nodes = {}  # id → properties
        self.edges = []  # (from, to, type, properties)
        self.adj = defaultdict(list)
    
    def add_node(self, node_id, **props):
        self.nodes[node_id] = props
    
    def add_edge(self, src, dst, rel_type, **props):
        self.edges.append((src, dst, rel_type, props))
        self.adj[src].append((dst, rel_type))
    
    def friends_of_friends(self, node_id):
        """2-hop traversal: find friends of friends."""
        friends = {dst for dst, rel in self.adj[node_id] if rel == "FRIENDS"}
        fof = set()
        for f in friends:
            for dst, rel in self.adj[f]:
                if rel == "FRIENDS" and dst != node_id and dst not in friends:
                    fof.add(dst)
        return fof

gdb = GraphDB()
for name in ["Alice", "Bob", "Charlie", "Diana", "Eve"]:
    gdb.add_node(name, type="Person")

gdb.add_edge("Alice", "Bob", "FRIENDS")
gdb.add_edge("Alice", "Charlie", "FRIENDS")
gdb.add_edge("Bob", "Diana", "FRIENDS")
gdb.add_edge("Charlie", "Eve", "FRIENDS")
gdb.add_edge("Bob", "Eve", "FRIENDS")

print("  Graph DB — Social Network:")
for src, dst, rel, _ in gdb.edges:
    print(f"    ({src}) -[{rel}]→ ({dst})")

fof = gdb.friends_of_friends("Alice")
print(f"\n  Friends of friends of Alice: {fof}")
print("  Cypher: MATCH (a:Person)-[:FRIENDS]->()-[:FRIENDS]->(fof)")
print("          WHERE a.name='Alice' RETURN fof")
print("\n  💡 Graph: traversals are O(edges) not O(rows). SQL JOINs would be O(n²)!")

In [ ]:
# ============================================
# 5. SQL vs NoSQL Decision Guide
# ============================================

print("""
  ┌────────────────────┬────────────────────────┬────────────────────────┐
  │ Factor              │ SQL                    │ NoSQL                  │
  ├────────────────────┼────────────────────────┼────────────────────────┤
  │ Schema              │ Fixed (rigid)          │ Flexible (dynamic)     │
  │ Relationships       │ JOINs (strong)         │ Embed/denormalize      │
  │ Scaling             │ Vertical (+RAM/CPU)    │ Horizontal (+nodes)   │
  │ Consistency         │ Strong (ACID)          │ Eventual (BASE)       │
  │ Query language      │ SQL (standardized)     │ Varies by DB          │
  │ Transactions        │ Multi-table ACID       │ Limited (per-doc)     │
  │ Best for reads      │ Complex analytics      │ Simple key lookups    │
  │ Best for writes     │ Moderate               │ Very high throughput  │
  └────────────────────┴────────────────────────┴────────────────────────┘

  Decision Flowchart:
  ┌─────────────────────────────────────────────────────────────────────┐
  │ Need ACID transactions?        → SQL (PostgreSQL, MySQL)           │
  │ Caching / sessions?            → Key-Value (Redis)                 │
  │ Flexible schema / JSON docs?   → Document (MongoDB)                │
  │ Write-heavy time-series?       → Column-Family (Cassandra)        │
  │ Complex relationships?         → Graph (Neo4j)                     │
  │ Need search / full-text?       → Search Engine (Elasticsearch)    │
  │ Not sure?                      → Start with PostgreSQL (safe bet) │
  └─────────────────────────────────────────────────────────────────────┘
""")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Name 4 types of NoSQL? | Key-Value, Document, Column-Family, Graph |
| 2 | SQL vs NoSQL when? | SQL: ACID, complex queries, joins. NoSQL: scale, flexibility, speed |
| 3 | MongoDB data modeling? | Embed for 1:1/1:few. Reference for 1:many/M:N. Model for access patterns |
| 4 | Cassandra partition key? | Determines data distribution. All queries MUST include partition key |
| 5 | When to use graph DB? | Deep relationship traversals (social, fraud, recommendations, knowledge graphs) |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Key-Value: O(1), caching, sessions (Redis)           │
## │  • Document: flexible schema, embed data (MongoDB)      │
## │  • Column-Family: write-heavy, time-series (Cassandra)  │
## │  • Graph: relationship traversals (Neo4j)               │
## │  • Default to SQL unless you have a specific NoSQL need │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **CAP Theorem** — Consistency, Availability, Partition Tolerance